# 제로 에너지 드링크 HPLC 분석 — Colab 실행판

터미널 없이 브라우저에서 돌리는 버전입니다. 크롬북에서도 그대로 됩니다.

**위에서부터 셀 왼쪽의 ▶ 버튼을 순서대로 누르시면 됩니다.**

---

### ⚠ 먼저 읽을 것

이 노트북에는 **모의 데이터 생성기**가 들어 있습니다. 실험 전에 코드가 도는지
확인하고 예상치를 잡는 용도입니다.

모의 데이터로 만든 숫자를 보고서에 실측값으로 적으면 **데이터 조작**입니다.
모의 데이터로 돌리면 결과 파일과 그림에 `SIMULATED` 표시가 자동으로 박히니
지우지 마세요. 보고서에 넣고 싶으면 "실험 전 모의 계산으로 예측한 값"이라고
밝히면 됩니다.

## 1단계 — 준비물 설치

분석 라이브러리와 한글 폰트를 깔습니다. 1~2분 걸립니다.

폰트를 까는 이유는 matplotlib이 한글 폰트 없이 그래프를 그리면 글자가 전부
네모(□□□)로 나오기 때문입니다. 그대로 보고서에 붙였다가 낭패를 봅니다.

In [ ]:
!pip install -q numpy pandas matplotlib scipy
!apt-get install -y -qq fonts-nanum > /dev/null 2>&1
!fc-cache -f > /dev/null 2>&1

import matplotlib.font_manager as fm
fm._load_fontmanager(try_read_cache=False)

installed = {f.name for f in fm.fontManager.ttflist}
print('한글 폰트 사용 가능:', 'NanumGothic' in installed)
print('준비 완료')

## 2단계 — 분석 코드 가져오기

아래 셀을 실행하면 **파일 선택** 버튼이 나옵니다.

먼저 브라우저에서 코드를 내려받으세요.

1. https://github.com/3407-png/energy-drink-hplc 접속
2. 초록색 **Code** 버튼 → **Download ZIP**
3. 아래 셀 실행 → 방금 받은 ZIP 파일 선택

> 저장소를 **Public**으로 바꾸면 이 과정 없이 아래 한 줄로 끝납니다.
> `!git clone https://github.com/3407-png/energy-drink-hplc`

In [ ]:
import os, sys, glob, zipfile, shutil
from google.colab import files

os.chdir('/content')

# 이미 받아 둔 게 있으면 지우고 새로 푼다
for old in glob.glob('/content/energy-drink-hplc*'):
    shutil.rmtree(old, ignore_errors=True)

print('GitHub에서 받은 ZIP 파일을 선택하세요.')
uploaded = files.upload()

zip_name = next(n for n in uploaded if n.lower().endswith('.zip'))
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall('/content')

# hplc 패키지가 들어 있는 폴더를 찾아 작업 폴더로 삼는다
pkg = glob.glob('/content/**/hplc/__init__.py', recursive=True)
if not pkg:
    raise SystemExit('hplc 폴더를 찾지 못했습니다. 올바른 ZIP인지 확인하세요.')

PROJECT = os.path.dirname(os.path.dirname(pkg[0]))
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)

print('작업 폴더:', PROJECT)
print('파일:', sorted(os.listdir(PROJECT)))

## 3단계 — 실험 설계 점검

실험 전에 확인할 것들을 출력합니다. 파일은 만들지 않고 화면에만 나옵니다.

- 두 성분의 예상 머무름 시간과 머무름 계수 k'
- 첨가 농도(0/20/40/60 ppm)가 시료 농도에 비해 적절한지
- 표준액을 몇 mL씩 넣어야 하는지
- 놓치기 쉬운 전처리 실수

In [ ]:
# --- 프로젝트 폴더로 이동 (셀을 어떤 순서로 실행해도 되도록) ---
import os, glob
_pkg = glob.glob('/content/**/hplc/__init__.py', recursive=True)
assert _pkg, '코드를 아직 못 찾았습니다. 2단계(코드 가져오기)를 먼저 실행하세요.'
os.chdir(os.path.dirname(os.path.dirname(_pkg[0])))

!python -m hplc design-check

## 4단계 — 모의 데이터로 전 과정 돌려보기

실험 데이터가 아직 없을 때, 코드가 제대로 도는지 확인하고 예상 결과를 봅니다.

**여기서 나오는 숫자는 실측값이 아닙니다.**

In [ ]:
# --- 프로젝트 폴더로 이동 (셀을 어떤 순서로 실행해도 되도록) ---
import os, glob
_pkg = glob.glob('/content/**/hplc/__init__.py', recursive=True)
assert _pkg, '코드를 아직 못 찾았습니다. 2단계(코드 가져오기)를 먼저 실행하세요.'
os.chdir(os.path.dirname(os.path.dirname(_pkg[0])))

!python -m hplc demo

## 5단계 — 그림 보기

생성된 그래프를 노트북 안에서 바로 확인합니다.

In [ ]:
# --- 프로젝트 폴더로 이동 (셀을 어떤 순서로 실행해도 되도록) ---
import os, glob
_pkg = glob.glob('/content/**/hplc/__init__.py', recursive=True)
assert _pkg, '코드를 아직 못 찾았습니다. 2단계(코드 가져오기)를 먼저 실행하세요.'
os.chdir(os.path.dirname(os.path.dirname(_pkg[0])))

from IPython.display import Image, display, Markdown
import glob, os

figs = sorted(glob.glob('output/*.png'))
if not figs:
    print('그림이 없습니다. 4단계를 먼저 실행하세요.')
for f in figs:
    display(Markdown(f'### {os.path.basename(f)}'))
    display(Image(filename=f))

## 6단계 — 보고서 읽기

표를 노트북 안에서 바로 봅니다. 보고서 7장에 그대로 옮겨 쓸 수 있는 형태입니다.

In [ ]:
# --- 프로젝트 폴더로 이동 (셀을 어떤 순서로 실행해도 되도록) ---
import os, glob
_pkg = glob.glob('/content/**/hplc/__init__.py', recursive=True)
assert _pkg, '코드를 아직 못 찾았습니다. 2단계(코드 가져오기)를 먼저 실행하세요.'
os.chdir(os.path.dirname(os.path.dirname(_pkg[0])))

from IPython.display import Markdown, display
import glob

reports = glob.glob('output/*report.md')
if reports:
    display(Markdown(open(reports[0], encoding='utf-8').read()))
else:
    print('보고서가 없습니다. 4단계를 먼저 실행하세요.')

---

# 실측 데이터 분석

## 7단계 — 이미 들어 있는 실측 데이터 바로 돌리기

실험에서 나온 피크 면적이 `data/measured_peak_areas.csv` 에 이미 들어 있습니다.
업로드 없이 아래 셀만 실행하면 됩니다.

실행 후 **그림·보고서 셀(5·6단계)을 다시 실행**하면 실측 결과를 볼 수 있습니다.
이번엔 `SIMULATED` 표시가 없습니다.


In [ ]:
# --- 프로젝트 폴더로 이동 (셀을 어떤 순서로 실행해도 되도록) ---
import os, glob
_pkg = glob.glob('/content/**/hplc/__init__.py', recursive=True)
assert _pkg, '코드를 아직 못 찾았습니다. 2단계(코드 가져오기)를 먼저 실행하세요.'
os.chdir(os.path.dirname(os.path.dirname(_pkg[0])))

!python -m hplc analyze data/measured_peak_areas.csv

---

# 실험 당일 이후

## 8단계 — 빈 입력표 받기

실행하면 `peak_areas_template.csv`가 크롬북으로 내려받아집니다.

구글 스프레드시트로 열어서 **`peak_area` 열**을 채우세요. 피크가 안 나온 칸은
비워두지 말고 `0`을 적습니다. `retention_min` 열도 채우면 피크를 잘못 잡았는지
자동으로 점검해 줍니다.

다 채우면 **파일 → 다운로드 → 쉼표로 구분된 값(.csv)** 으로 내려받으세요.

In [ ]:
# --- 프로젝트 폴더로 이동 (셀을 어떤 순서로 실행해도 되도록) ---
import os, glob
_pkg = glob.glob('/content/**/hplc/__init__.py', recursive=True)
assert _pkg, '코드를 아직 못 찾았습니다. 2단계(코드 가져오기)를 먼저 실행하세요.'
os.chdir(os.path.dirname(os.path.dirname(_pkg[0])))

from google.colab import files

!python -m hplc template
files.download('data/peak_areas_template.csv')

## 9단계 — 실측 데이터 분석

채워 넣은 CSV를 올려서 분석합니다. 실행하면 파일 선택 버튼이 나옵니다.

분석이 끝나면 **5단계·6단계 셀을 다시 실행**해서 실측 그림과 보고서를
보시면 됩니다. 이번엔 `SIMULATED` 표시가 없습니다.
> **검량선(증류수 바탕) 데이터가 없어도 됩니다.** 표준물 첨가법의 농도
> 역산은 검량선을 쓰지 않으므로 결과가 그대로 나옵니다. 다만 **변환 상수**는
> 검량선이 있어야 계산되므로 그 칸만 비게 됩니다.
>
> 변환 상수까지 구하려면 둘 중 하나를 하세요.
> - CSV에 `group=calib` 행(농도 3수준 이상)을 추가한다
> - 왼쪽 폴더 아이콘에서 `hplc/config.py` 를 열어 `EXTERNAL_CALIBRATION` 에
>   `{"caffeine": (기울기, y절편), "sodium_benzoate": (기울기, y절편)}` 을 적는다


In [ ]:
# --- 프로젝트 폴더로 이동 (셀을 어떤 순서로 실행해도 되도록) ---
import os, glob
_pkg = glob.glob('/content/**/hplc/__init__.py', recursive=True)
assert _pkg, '코드를 아직 못 찾았습니다. 2단계(코드 가져오기)를 먼저 실행하세요.'
os.chdir(os.path.dirname(os.path.dirname(_pkg[0])))

import shutil, os
from google.colab import files

# 모의 결과가 섞이지 않도록 이전 출력물을 지운다
shutil.rmtree('output', ignore_errors=True)

print('실험 데이터를 채운 CSV 파일을 선택하세요.')
uploaded = files.upload()
csv_name = next(n for n in uploaded if n.lower().endswith('.csv'))

# Colab 은 같은 이름이 이미 있으면 'xxx (1).csv' 로 저장해 버린다.
# 그러면 방금 올린 파일이 아니라 예전 파일을 분석하게 되므로,
# 업로드된 내용을 항상 같은 이름으로 직접 써 준다.
MEASURED = 'measured_peak_areas.csv'
with open(MEASURED, 'wb') as fh:
    fh.write(uploaded[csv_name])
print(f'{csv_name} -> {MEASURED} ({len(uploaded[csv_name]):,} bytes)')

!python -m hplc analyze "{MEASURED}"

## 10단계 — 결과 전부 내려받기

보고서, 표, 그림을 ZIP 하나로 묶어 크롬북에 저장합니다.

In [ ]:
# --- 프로젝트 폴더로 이동 (셀을 어떤 순서로 실행해도 되도록) ---
import os, glob
_pkg = glob.glob('/content/**/hplc/__init__.py', recursive=True)
assert _pkg, '코드를 아직 못 찾았습니다. 2단계(코드 가져오기)를 먼저 실행하세요.'
os.chdir(os.path.dirname(os.path.dirname(_pkg[0])))

import shutil, os
from google.colab import files

if not os.path.isdir('output'):
    print('결과가 없습니다. 4단계 · 7단계 · 9단계 중 하나를 먼저 실행하세요.')
else:
    shutil.make_archive('hplc_results', 'zip', 'output')
    files.download('hplc_results.zip')

---

## 조건이 또 바뀌면

`hplc/config.py` 한 파일만 고치면 코드 전체가 따라옵니다. Colab 왼쪽 폴더
아이콘에서 `hplc/config.py`를 더블클릭하면 편집기가 열립니다. 고친 뒤 2단계를
건너뛰고 3단계부터 다시 실행하세요.

자주 고치게 되는 값:

| 위치 | 무엇 |
|---|---|
| `HPLCConditions` | 이동상 비율, 유량, 오븐 온도, 검출 파장, run time |
| `expected_rt_min` | 실측 머무름 시간 (첫 크로마토그램 나오면 꼭 교체) |
| `DRINKS` | 시료 이름, 제공량, 라벨 카페인 표시량 |
| `PrepConditions` | 희석배수, 첨가 농도, 주입 횟수 |
| `ACTIVE_COMPOUNDS` | 분석 성분 목록 |

> Colab에서 고친 내용은 세션이 끊기면 사라집니다. 계속 쓸 값이면 GitHub의
> `config.py`도 같이 고쳐 두세요.